In [ ]:
import os
import smtplib

from email.mime.text import MIMEText

from dotenv import load_dotenv
from getpass import getpass

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

In [ ]:
os.environ['OPENAI_API_KEY'] = getpass('Voer je OpenAI API key in: ')
os.environ['OUTLOOK_EMAIL'] = input('Voer je Outlook email adres in: ')
os.environ['OUTLOOK_PASSWORD'] = getpass('Voer je Outlook wachtwoord in: ')

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
email_address = os.getenv('OUTLOOK_EMAIL')
email_password = os.getenv('OUTLOOK_PASSWORD')

model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

In [ ]:
llm = ChatOpenAI(
    model=model,
    api_key=api_key,
)

In [ ]:
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """
    Send an email using Outlook.

    Parameters:
    - to: recipient email address
    - subject: email subject
    - body: email body
    """
    sender = os.getenv("OUTLOOK_EMAIL")
    password = os.getenv("OUTLOOK_PASSWORD")

    if not sender or not password:
        msg = "Error: OUTLOOK_EMAIL or OUTLOOK_PASSWORD not set."
        print(msg)
        return msg

    msg = MIMEText(body)
    msg["Subject"] = subject
    msg["From"] = sender
    msg["To"] = to

    try:
        with smtplib.SMTP("smtp.office365.com", 587, timeout=10) as server:
            server.starttls()
            server.login(sender, password)
            server.sendmail(sender, to, msg.as_string())
        result = f"Email successfully sent to {to}"
        print(result)
        return result

    except smtplib.SMTPAuthenticationError as e:
        result = f"SMTPAuthenticationError: {e}\nTip: use an App Password if MFA is enabled (https://account.microsoft.com/security)"
        print(result)
        return result
    except Exception as e:
        result = f"Error sending email ({type(e).__name__}): {e}"
        print(result)
        return result

tools = [send_email]

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful assistant.

    Use the email tool whenever the user wants to send an email.
    """,
)

In [ ]:
task = """
Send an email to mark.christianen@d-data.nl.

Subject:
Meeting Tomorrow

Body:
Hi,

Just a reminder that we have a meeting tomorrow at 10:00.

Best regards,
Mark
"""

In [ ]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": task}
    ]
})
print(response["messages"][-1].content)